# Winner public LightGBM baseline
[Konstantin Yakovlev 공개 학습 코드](https://www.kaggle.com/kyakovlev/ieee-lgbm-with-groupkfold-cv)를 실제 출처로 사용한다.
[원본 CSV minification](https://www.kaggle.com/kyakovlev/ieee-data-minification) → [공개 FE](https://www.kaggle.com/kyakovlev/ieee-fe-with-some-eda) → 월별 GroupKFold 학습 경로다.
최종 우승 CAT/LGB 모델 전체 코드로 확인된 것은 아닌 **우승자 본인의 공개 baseline**이다. 17위 등 다른 참가자 구현을 사용하지 않는다.
공통 FE는 `data/winner_fe.py`에 분리했고 학습은 이 노트북에서 직접 한다. 전처리 pickle이나 EDA 실행을 선행할 필요가 없다.

## 이 노트북을 읽는 방법

코드를 실행하기 전에 바로 위의 설명을 읽어보세요. **어떤 질문을 푸는지 → 작은 예시로 계산 → 실제 코드의 변수와 연결 → 출력 해석** 순서로 설명합니다.
코드 아래의 관찰은 이미 저장된 실행 결과를 읽는 안내입니다. '해볼 실험'은 아직 실행하지 않은 제안이며, 실제 결과와 구분했습니다.

처음 읽을 때 함수 이름을 모두 외울 필요는 없습니다. 새 피처를 만날 때마다 **'이 숫자는 무엇을 요약하며, 예측할 때도 알 수 있는가?'**를 물어보세요.
EDA에서 찾은 차이가 모델 성능 개선을 뜻하지는 않습니다. 실험 노트북에서는 **검증 데이터를 정한 뒤 구성 요소 하나씩 비교**해야 개선의 근거를 얻습니다.

처음에는 `01_eda_report` → GitHub `baseline`을 읽고, 이후 `02_winner_eda` → `winner_xgb` → `winner_lgbm` → `winner_catboost` → `winner_blend`로 이어가세요.
우승자 공개 baseline과 최종 우승 제출 전체는 구분합니다.

설명을 보강하면서 학습 코드·기존 표·그래프·실행 범위를 유지했습니다. 이 노트북의 **저장된 출력**과 설정의 **다음 실행 기본값**이 다를 수 있으므로 첫 실행 범위와 metrics를 먼저 확인하세요.


### 처음 만나는 용어는 여기서 잠깐 확인하세요

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 피처(feature) | 모델에 입력할 거래의 정보. 원본 열과 새로 계산한 열 모두 포함 |
| NaN / 결측 | 값이 관측되지 않음. 실제 숫자 0과 다른 상태 |
| fold / validation | 교차검증의 한 분할 / 그 분할에서 평가용으로 제외한 데이터 |
| OOF | 각 train 행을 그 행 없이 학습한 모델로 예측해서 모은 값 |
| smoke | 전체 학습 전에 축소 데이터·rounds로 실행 흐름을 확인하는 실험 |
| leaf | tree에서 조건을 따라 내려간 끝의 구역. 그 구역에 들어온 행에 같은 보정을 줌 |

**제거 실험(ablation)**은 피처나 기법 하나를 뺀 모델을 같은 검증에서 비교하는 방법입니다. '있을 때 좋았으니 도움이 된다'에서 한 걸음 더 나아가 실제 기여를 확인하려는 실험입니다.

## 1. 원본 CSV와 공개 FE

### 같은 원본에서 LightGBM용 실험을 시작합니다

이 노트북은 우승자의 공개 LightGBM baseline입니다. CatBoost 노트북과 `build_features`를 공유하지만 CatBoost용 지배값 flag/범주 복원/UID 재포함은 수행하지 않습니다.
따라서 현재 점수 차이를 '모델 이름 하나의 차이'로 해석할 수 없습니다. 모델만 비교하려면 같은 피처·분할·학습 예산을 별도로 맞춰야 합니다.

`NROWS=None`이면 전체 원본 CSV이고, 지정하면 월별 일정 비율 표본을 사용합니다. `target_keys`는 ProductCD/M4의 그룹 키를 복사합니다.
정답을 이용하는 평균은 뒤 CV 안에서만 만들고, 정답을 쓰지 않는 배치 집계는 이 앞 단계에서 만듭니다. 둘의 정보 사용 범위를 구분하며 읽어보세요.

### `build_features()` 안에서 만드는 피처도 여기서 공부합니다

두 모델이 같은 FE를 쓰므로 구현만 `data/winner_fe.py`에 모았습니다. 코드 아래의 import를 몰라도 흐름을 이해할 수 있도록 실제 단계와 예시를 짚어봅니다.

**1. 원본 CSV와 값의 표현.** train/test transaction과 identity를 읽고 ID로 결합합니다. card4/card6/ProductCD/M4는 공동 빈도로 바꾸고 M의 T/F는 1/0으로 바꿉니다.
범주 A/B/C가 각각 100/10/10번이면 빈도는 흔함을 표현하지만 B/C는 같은 10이 됩니다. **빈도는 범주 정체성의 완전한 대체가 아닙니다.**
test의 `isFraud=0`은 표 구조를 맞추는 임시 값입니다. 실제 정답도, 정상 거래라는 판정도 아닙니다. 모델 입력에서 제외하고 학습 정답은 train에서만 가져옵니다.
float32는 메모리를 줄이는 수치 표현입니다. 표준화나 새로운 예측 정보가 아니며, original float16 축소보다 정밀도를 더 유지하는 변경입니다.

**2. 시간과 활동 비율.** 월/주/일/시간과 휴일 표시를 만든 뒤 `DT_M/DT_W/DT_D`는 집계 기준으로 쓰고 모델 입력에서 제외합니다.
어느 은행 그룹의 하루 거래가 100건이라도 그날 전체가 200건인지 20,000건인지에 따라 의미가 다릅니다. `time_frequency`는 **그룹 거래 수 / 그 시간 블록 전체 거래 수**를 만듭니다.
은행 그룹의 평균/최빈 활동 시간에서 현재 시간을 뺀 피처도 만듭니다. 평소 오후인데 새벽인 거래를 표현할 후보입니다. 시계의 23시와 0시가 실제로는 가까운 점은 단순 차이가 잘 표현하지 못합니다.

**3. 희귀 카드 처리와 여러 UID.** 원문대로 train/test 양쪽에 존재하지 않는 카드 코드를 결측으로 만들고 card1 중 2건 이하도 비웁니다.
이는 드문 식별자를 암기하는 일을 줄이려는 선택입니다. 반면 희귀한 카드의 정보도 잃으므로 항상 이득인 규칙은 아닙니다. test 전체 코드 목록을 볼 수 있는 대회 배치 조건을 사용합니다.
`uid→uid2→uid3→uid4/5`는 카드, 추가 카드 속성, 주소, 이메일을 점점 더 붙입니다. 좁게 묶으면 개인화되지만 표본이 줄고, 넓게 묶으면 통계가 안정적인 대신 여러 고객이 섞입니다.

**4. 그룹 평균과 변동.** 같은 UID의 금액 [10,10,40]의 평균은 20이고 표본 std는 약 17.32입니다.
이 std는 `sqrt(((10-20)²+(10-20)²+(40-20)²)/(3-1))`입니다. 평균에서 얼마나 벗어나는지 제곱으로 모아 표본 변동을 요약합니다.
현재 금액 40과 평균 20을 같이 주면 모델이 '이 그룹에서 큰 금액인가'를 배울 수 있습니다. std는 활동의 다양성을 표현합니다. 1건 그룹의 std는 0이 아니라 NaN입니다.
`aggregate`는 D와 금액의 mean/std를 card/UID/bank 그룹에 붙입니다. **정답 평균이 아니라 입력 피처의 평균**이므로 target encoding과 다릅니다.

**5. 같은 시기 안에서의 정규화.** `normalize`는 train과 test 각각의 일/주/월 안에서 min-max와 표준 점수를 만듭니다.
같은 기간의 값 [10,20,30]에서 30의 min-max는 (30-10)/(30-10)=1입니다. 평균 20, 표본 std 10이면 표준 점수는 (30-20)/10=1입니다.
같은 기간에서 상대적으로 높은 값을 표현하지만 이 값은 UID 안의 비교가 아닙니다. tree에 단순 스케일링이 필수라서가 아니라 **시간 블록에 따른 상대 위치라는 새 정보**를 만들기 위해 씁니다. 값이 모두 같으면 분모가 0이라 결측이 생길 수 있습니다.

**6. D와 금액/C 변환의 순서.** 앞의 D 그룹 집계는 clip 이전 값으로 만듭니다. 이후 D의 음수를 0으로 clip하고 D8/D9의 결측/소수 관계 피처를 만든 뒤 정규화합니다. D1/D2는 train 최대값으로 나눈 별도 scaled 열을 만듭니다. 원본 D를 자기 빈도 값으로 바꾸기 때문에 변환 이후 D 열은 '원래 일수'가 아닙니다.
금액은 5,000으로 clip하고 집계/정규화/상품×금액 빈도를 만든 후 `log1p`로 원본 금액을 바꿉니다. clip은 정보 일부를 버리고 log는 양수 범위 순서를 유지하며 큰 간격을 줄입니다.
C에는 빈도를 추가하고 마지막 train 월의 최대값으로 상한을 둡니다. 집계가 **원 단위 금액인지, log 금액인지**는 만들어진 순서에 달려 있습니다.

**7. 디바이스/ID와 문자열.** identity의 Found/T 같은 값을 매핑하고, 화면 해상도를 가로/세로로 나누며, 기기/OS/브라우저에서 문자와 숫자 부분을 분리합니다.
이를 빈도로 요약하면 희귀한 버전/기기를 표현할 수 있지만 정확한 버전 숫자가 서열을 의미하지는 않습니다. 남은 문자열은 공동 label encoding 후 category로 둡니다.
ProductCD/M4 target mean은 공통 FE에서 미리 만들지 않고 **각 모델의 fit fold 정답**으로 계산합니다.

이 FE는 여러 입력을 동시에 사용해 **전체 배치에서의 문맥**을 만듭니다. 정답이 없다고 자동으로 실시간에 쓸 수 있는 것은 아닙니다. 하루 전체 빈도나 미래 거래까지 포함한 평균은 그 거래 순간에는 아직 모를 수 있습니다.

In [1]:
import gc, json, logging, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'source.json').exists())
sys.path.insert(0, str(ROOT))
from data.loader import get_data_dir
from data.winner_fe import CAT_ORIGINAL, build_features
NROWS = int(os.environ['IEEE_WINNER_NROWS']) if os.getenv('IEEE_WINNER_NROWS') else None
FOLDS = int(os.getenv('IEEE_WINNER_FOLDS', '6'))
TAG = os.getenv('IEEE_WINNER_TAG', 'smoke' if NROWS else 'full')
OUT = ROOT / 'experiments/winner_lgbm/outputs' / TAG
OUT.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger('winner_lgbm')
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.addHandler(logging.FileHandler(OUT / 'run.log', mode='w', encoding='utf-8'))
logger.addHandler(logging.StreamHandler(sys.stdout))
started = time.perf_counter()
train, test, remove, original = build_features(NROWS)
target_keys = {col: (train[col].copy(), test[col].copy()) for col in ['ProductCD', 'M4']}
logger.info('FE train=%s test=%s monthly rows=%s', train.shape, test.shape, train.DT_M.value_counts().sort_index().to_dict())

FE train=(30000, 790) test=(30002, 790) monthly rows={12: 6976, 13: 4703, 14: 4370, 15: 5163, 16: 4250, 17: 4538}


## 2. 모델별 공개 코드의 준비 단계

### LightGBM은 작은 tree들을 어떤 순서로 더할까요?

### 먼저, 여러 tree를 왜 순서대로 더할까요?

한 tree는 '금액이 크고, 특정 기기이고, 어떤 카드 그룹인가'처럼 조건을 나눠 거래를 구분합니다. 여러 조건의 **상호작용**을 표현할 수 있습니다.
Gradient boosting은 새 tree를 독립적으로 평균내는 방식과 다릅니다. 앞의 예측이 놓친 방향을 다음 tree가 보정합니다.

이진 분류에서 예측 확률 p와 정답 y의 log loss는 `-[y log(p)+(1-y)log(1-p)]`입니다.
사기인데 p=.1이면 큰 벌점을 받고, 사기인데 p=.9이면 작은 벌점을 받습니다. logit 점수에 대한 음의 기울기는 `y-p`입니다.
사기 y=1,p=.1은 +.9 방향, 정상 y=0,p=.9는 -.9 방향으로 보정하도록 새 tree를 학습합니다. 실제 leaf 값은 여러 행의 기울기/가중치 등을 모아 구합니다.

개념적으로 `새 점수 = 이전 점수 + learning_rate × 새 tree 보정`이며 확률은 점수를 sigmoid로 바꿔 얻습니다.
작은 learning rate는 조금씩 고치는 대신 더 많은 tree가 필요합니다. **rounds를 크게 적었다고 반드시 그 수만큼 학습하는 것도, 최적 튜닝을 마친 것도 아닙니다.**

여기서 loss는 학습 방향을 정하고 AUC는 검증에서 순위를 평가합니다. **AUC를 지표로 적었어도 tree가 직접 AUC를 미분해 학습하는 것은 아닙니다.**
트리의 깊이/leaf 수는 복잡도, 행/열 표본 추출은 다양성과 과적합을, early stopping은 학습을 끝낼 시점을 조절합니다.
이 설명의 수식은 원리를 이해하기 위한 것이며 실제 새 tree는 단일 거래의 오차만으로 만들어지지 않습니다.

여기서 gradient(기울기)는 loss를 줄이려면 예측을 어느 방향으로 고칠지 알려줍니다.
확률 p를 `log(p/(1-p))`로 바꾼 점수를 **logit**이라 부릅니다. p=.5이면 0, p=.9이면 약 2.20입니다.
**Sigmoid**는 `1/(1+exp(-점수))`로 점수를 다시 0~1 확률로 바꿉니다. Tree 보정은 이 점수 공간에서 더하고 마지막에 확률로 읽습니다.


### LightGBM의 tree 모양: 개선 폭이 큰 leaf부터 나눕니다

두 leaf 중 한쪽을 더 나누면 loss가 8 줄고, 다른 쪽은 1 줄어든다고 해봅시다. Leaf-wise는 현재 가장 큰 개선을 낼 쪽부터 나눕니다.
한쪽 경로가 깊어져 복잡한 예외를 잘 학습할 수 있지만 작은 표본·희귀 그룹에서는 이를 외울 수도 있습니다.
`num_leaves=256`은 최대 leaf 수이지, 모든 경로의 depth가 8이라는 뜻이 아닙니다. `max_depth=-1`은 아래 설정에서 깊이 상한을 두지 않는다는 뜻입니다.
Histogram은 수치 값을 bin으로 묶어 분기 후보 탐색을 빠르게 합니다. 원래 모든 소수점 차이를 개별 후보로 검사하는 것과 속도·표현의 tradeoff가 있습니다.
[공식 tree/histogram 설명](https://lightgbm.readthedocs.io/en/stable/Features.html)을 참고하세요.

Tree는 이미 값의 크기에 따라 나눌 수 있어 단순 표준화가 선형 모델처럼 필수는 아닙니다.
공통 FE의 기간별 z-score/비율은 스케일만 바꾸려는 것이 아니라 **같은 기간의 다른 거래에 비해 상대적으로 큰가**라는 새로운 정보를 줍니다.
pandas `category` 열은 기본 자동 인식으로 범주형 분기를 사용합니다. 그냥 빈도 숫자로 바뀐 열은 수치 분기에 쓰이므로 두 표현을 혼동하지 마세요.

| 설정 | 무엇을 조절하나요? | 아직 실행하지 않은 비교 |
|---|---|---|
| learning_rate=.007 | tree 보정 크기 | rounds를 충분히 두고 학습 곡선 확인 |
| num_leaves=256 | tree가 만들 수 있는 세부 구역 수 | 64/128/256에서 월별 안정성 확인 |
| max_depth=-1 | 분기 깊이 상한 없음 | leaf 수와 최소 leaf 표본을 함께 검토 |
| feature_fraction=.5 | tree마다 사용하는 열 비율 | 유사 UID 집계가 많은 상황의 영향 확인 |
| bagging_fraction=.7, freq=1 | 매 iteration 행 일부 사용 | 희귀 사기와 표본 크기별 변화 확인 |
| max_bin=255 | histogram 해상도 | 속도와 드문 값 구분의 균형 확인 |

위 비교를 수행한 출력은 아직 없습니다. 256 leaf는 저자의 공개 설정이며 최적임을 입증한 로컬 튜닝 결과가 아닙니다.
`min_data_in_leaf`를 늘리는 실험은 매우 작은 그룹을 외우는 것을 줄일 수 있으나 유용한 희귀 사기도 놓칠 수 있습니다.
[공식 튜닝 안내](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html)와 함께 검증 결과로 판단하세요.

In [2]:
import lightgbm as lgb
features = [c for c in train if c not in remove]
ROUNDS = int(os.getenv('IEEE_WINNER_ROUNDS', '10000'))
params = dict(objective='binary', boosting_type='gbdt', metric='auc', learning_rate=.007,
              num_leaves=256, max_depth=-1, tree_learner='serial', feature_fraction=.5,
              bagging_freq=1, bagging_fraction=.7, max_bin=255, verbosity=-1, seed=42, num_threads=8)
logger.info('features=%d params=%s rounds=%d', len(features), params, ROUNDS)

features=772 params={'objective': 'binary', 'boosting_type': 'gbdt', 'metric': 'auc', 'learning_rate': 0.007, 'num_leaves': 256, 'max_depth': -1, 'tree_learner': 'serial', 'feature_fraction': 0.5, 'bagging_freq': 1, 'bagging_fraction': 0.7, 'max_bin': 255, 'verbosity': -1, 'seed': 42, 'num_threads': 8} rounds=100


## 3. 월별 CV

### 모델 입력을 fold마다 만들고, 제외한 월을 예측합니다

이 루프에서는 `fit_x`의 행·라벨로 모델을 학습하고, `val_x`의 AUC로 early stopping 시점을 선택합니다.
target mean도 fit 라벨로만 계산합니다. 예측 전에 만든 정답 피처가 validation을 이미 봤는지 확인하는 것이 모델 종류보다 우선입니다.

### '이 범주는 사기가 얼마나 많았는가'를 입력으로 쓰는 방법

Target mean encoding은 학습 데이터에서 범주별 정답 평균을 구해 피처로 넣습니다. 학습 fold에서 M4=A의 라벨이 [0,0,1,0]이면 값은 .25입니다.
검증 A도 .25를 받고, 학습에 없던 B는 그 fold 전체 사기율을 받습니다. **검증 B의 정답이 1이어도 인코딩을 만들 때 알면 안 됩니다.**

그래서 코드는 **fold 분리 → fit 정답으로 mapping 작성 → fit/validation/test에 같은 mapping 적용** 순서입니다.
전체 train 정답으로 mapping을 만들고 CV하면 검증 정답 일부가 이미 피처에 들어가 점수가 낙관적으로 보일 수 있습니다.

주의할 구분이 하나 더 있습니다. 이 구현은 validation 정답은 제외하지만 fit 행의 인코딩에는 자기 정답도 평균의 일부로 들어갑니다.
범주가 1건뿐이면 평균이 정답 자체라 과적합하기 쉽습니다. **높은 cardinality에서는 내부 OOF 인코딩/leave-one-out/전체 평균으로 smoothing을 고려할 수 있습니다.** 여기서는 그 추가 기법을 구현하거나 효과를 검증한 것은 아닙니다.

### fold, OOF, test 평균을 구분해서 읽기

Fold는 교차검증의 한 번의 학습/평가 분할입니다. 월별 6-fold이면 매번 한 월을 검증에 두고 나머지 월로 학습합니다.
검증 월의 행은 자기 정답을 학습하지 않은 모델에서 예측을 받습니다. 그 값을 원래 행 위치에 모은 것이 **OOF(out-of-fold)**입니다.
OOF를 합쳐 AUC를 구하면 학습 행 재예측보다 일반화 평가에 가깝습니다. 다만 여기서는 early stopping도 같은 validation을 보므로 완전히 손대지 않은 최종 평가셋은 아닙니다.

`pred += fold_test_pred / FOLDS`는 test를 각 fold 모델로 예측해 평균냅니다. test에는 정답이 없어 AUC를 계산할 수 없습니다.
GroupKFold의 groups는 월입니다. **같은 월의 행이 fit/validation 양쪽에 들어가지는 않지만 같은 UID의 다른 월 거래는 들어갈 수 있습니다.**
또 검증이 과거 월이면 이후 월도 fit에 들어갑니다. 그러므로 이 검증을 미래 예측이나 고객 전체 미관측 검증으로 부르면 안 됩니다.
미래 예측은 시간 holdout, 새 고객 예측은 UID 그룹 분할 등으로 따로 질문해야 합니다. [GroupKFold 정의](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)를 참고하세요.

### `oof`와 `oof_scaled`는 왜 둘 다 저장할까요?

`oof[val_idx] = values`는 해당 행을 학습에 넣지 않은 모델의 예측을 원래 위치에 돌려놓습니다. 모든 fold가 끝나면 모든 train 행에 OOF 예측 하나씩 생깁니다.
반면 `pred += ... / FOLDS`는 여러 모델이 **같은 test 행**에 낸 예측을 평균냅니다. OOF와 test 평균은 만드는 방식이 다릅니다.

저자의 공개 코드는 각 validation fold에서 예측을 `(p-min)/(max-min)`으로 바꾼 뒤 합친 OOF도 평가합니다. 이 노트북은 그 결과를 `oof_scaled`로 별도 보존합니다.
한 fold 안에서는 순서가 유지되므로 그 fold의 AUC는 그대로지만, **서로 다른 fold 사이의 순서**는 바뀔 수 있습니다.
예를 들어 fold A의 범위 [.7,.9]에서 .7은 0이 되고, fold B의 [.1,.2]에서 .2는 1이 됩니다. 원래 .7>.2였던 비교가 뒤집힙니다.

따라서 pooled AUC의 변화는 모델을 다시 학습해서 생긴 개선이 아닙니다. 이 변환은 실제 사기 빈도에 맞춘 확률 보정(calibration)도 아닙니다.
아래 코드는 test 예측에 같은 min-max를 적용하지 않습니다. **raw OOF, fold별 AUC, source-scaled OOF를 따로 읽고**, source-scaled 점수만으로 모델을 선택하지 마세요.

### Early stopping이 알려주는 것과 알려주지 않는 것

`early_stopping(100)`은 validation AUC의 개선이 100 rounds 동안 없으면 학습을 멈추고 best iteration을 사용합니다.
rounds 상한이 100인 smoke에서는 이 대기 구간을 충분히 관찰하지 못합니다. 특히 learning_rate=.007은 한 tree의 변화가 작아 100 rounds만으로 성능 잠재력을 판단하기 어렵습니다.
하지만 낮은 점수를 전부 '학습 부족'으로 단정할 수도 없습니다. 피처·검증 월·과적합의 영향은 충분한 rounds에서 다시 비교해야 합니다.
`model.predict`는 early stopping으로 기록된 best iteration을 기본 사용합니다. `gain`은 해당 분기에 의해 줄인 학습 loss의 합을 fold 평균해 저장합니다.

In [3]:
y, groups = train.isFraud.astype('int8'), train.DT_M
oof, oof_scaled, pred = np.full(len(train), np.nan), np.full(len(train), np.nan), np.zeros(len(test))
importance, folds = np.zeros(len(features)), []
for fold, (fit_idx, val_idx) in enumerate(GroupKFold(FOLDS).split(train, y, groups)):
    assert set(groups.iloc[fit_idx]).isdisjoint(groups.iloc[val_idx])
    fit_x, val_x, test_x = train[features].iloc[fit_idx].copy(), train[features].iloc[val_idx].copy(), test[features].copy()
    for col, (train_key, test_key) in target_keys.items():
        if col in features:
            mapping = y.iloc[fit_idx].groupby(train_key.iloc[fit_idx]).mean()
            fit_x[col] = train_key.iloc[fit_idx].map(mapping).fillna(y.iloc[fit_idx].mean())
            val_x[col] = train_key.iloc[val_idx].map(mapping).fillna(y.iloc[fit_idx].mean())
            test_x[col] = test_key.map(mapping).fillna(y.iloc[fit_idx].mean())
    fit_x, val_x, test_x = [frame.replace([np.inf, -np.inf], np.nan) for frame in [fit_x, val_x, test_x]]
    model = lgb.train(params, lgb.Dataset(fit_x, label=y.iloc[fit_idx]), num_boost_round=ROUNDS,
                      valid_sets=[lgb.Dataset(val_x, label=y.iloc[val_idx])], callbacks=[lgb.early_stopping(100, verbose=False)])
    values = model.predict(val_x)
    pred += model.predict(test_x) / FOLDS
    importance += model.feature_importance(importance_type='gain') / FOLDS
    best_iteration = model.best_iteration

    oof[val_idx] = values
    spread = values.max() - values.min()
    oof_scaled[val_idx] = (values - values.min()) / spread if spread else 0
    folds.append({'fold': fold, 'months': sorted(map(int, groups.iloc[val_idx].unique())),
                  'auc': float(roc_auc_score(y.iloc[val_idx], values)), 'best_iteration': int(best_iteration)})
    logger.info('fold=%d months=%s AUC=%.6f best_iteration=%d', fold, folds[-1]['months'], folds[-1]['auc'], best_iteration)
    del model, fit_x, val_x, test_x
    gc.collect()
assert np.isfinite(oof).all() and np.isfinite(pred).all()
assert np.all((oof >= 0) & (oof <= 1)) and np.all((pred >= 0) & (pred <= 1))
display(pd.DataFrame(folds))

fold=0 months=[12] AUC=0.806292 best_iteration=95


fold=1 months=[15] AUC=0.851515 best_iteration=100


fold=2 months=[13] AUC=0.843895 best_iteration=97


fold=3 months=[17] AUC=0.880472 best_iteration=41


fold=4 months=[14] AUC=0.875902 best_iteration=55


fold=5 months=[16] AUC=0.880817 best_iteration=99


,fold,months,auc,best_iteration
0,0,[12],0.806292,95
1,1,[15],0.851515,100
2,2,[13],0.843895,97
3,3,[17],0.880472,41
4,4,[14],0.875902,55
5,5,[16],0.880817,99


### fold 표를 읽는 연습
현재 일부 best_iteration이 100입니다. CatBoost의 0-based 99와 표기 방식은 다르지만 여기서는 100-round 상한에 닿았다는 뜻입니다.
먼저 월별 AUC의 차이를 보고 그다음 상한을 확인하세요. 한 월의 점수가 낮다고 해당 월 데이터를 삭제하는 결론으로 바로 넘어가지 않습니다.

## 4. 산출물과 변경

### 성능 숫자를 남기고, 다음 질문을 하나 고릅니다

`metrics.json`에는 점수뿐 아니라 표본 크기·fold·rounds·버전이 같이 들어갑니다. 이 정보가 있어야 다음 실행과 공정하게 비교할 수 있습니다.
`oof.csv`의 month와 정답으로 월별 오류를 보고, 다음에는 ProductCD/identity/UID 매칭 여부 등 원본 정보를 ID로 붙여 오류가 집중되는 곳을 조사할 수 있습니다.
이는 **제안한 후속 분석**이며 아래 셀은 기록 저장만 합니다.

높은 gain 피처는 이 모델이 분기에 많이 활용한 정보입니다. 타깃의 원인이라는 뜻도, 높은 피처만 남기면 더 좋다는 뜻도 아닙니다.
유사한 집계 열이 서로 대체할 수 있으므로 **관련 피처 묶음을 제외한 비교**와 validation에서의 중요도 분석이 다음 질문이 됩니다.
CatBoost의 PredictionValuesChange와 정의가 달라 중요도 숫자를 모델끼리 직접 비교하지 않습니다.

`submission.reindex(sample)`은 확률을 해당 거래 ID 순서에 맞춥니다. 축소 실행의 sample_predictions는 전체 대회 제출 파일이 아닙니다.
**아직 실행하지 않은 순서:** 같은 검증에서 충분한 rounds 확보 → leaf/최소 표본 규제 비교 → UID 집계/기간 피처 묶음 제거 비교 → 마지막 월 holdout 확인.
튜닝 도중 validation을 계속 보면 그 validation에도 맞춰질 수 있으므로 최종 평가용 기간을 따로 남겨두는 설계도 고려하세요.

In [4]:
np.save(OUT / 'oof_train.npy', oof)
np.save(OUT / 'oof_scaled.npy', oof_scaled)
np.save(OUT / 'pred_test.npy', pred)
pd.DataFrame({'TransactionID': train.TransactionID, 'isFraud': y, 'oof': oof, 'oof_source_scaled': oof_scaled, 'month': groups}).to_csv(OUT / 'oof.csv', index=False)
pd.Series(importance, index=features, name='importance').sort_values(ascending=False).to_csv(OUT / 'feature_importance.csv')
submission = pd.Series(pred, index=test.TransactionID, name='isFraud')
if NROWS:
    submission.to_csv(OUT / 'sample_predictions.csv')
else:
    sample = pd.read_csv(get_data_dir() / 'sample_submission.csv').TransactionID
    assert submission.index.is_unique and set(sample) == set(submission.index)
    submission.reindex(sample).to_csv(OUT / 'submission.csv')
metrics = {'scope': 'monthly sampled smoke' if NROWS else 'full raw CSV public baseline',
           'train_rows': len(train), 'test_rows': len(test), 'features': len(features), 'folds': folds, 'params': params,
           'rounds': ROUNDS, 'target_mean': 'fold-local', 'numeric_precision': 'float32, not source float16',
           'oof_auc': float(roc_auc_score(y, oof)), 'source_scaled_oof_auc': float(roc_auc_score(y, oof_scaled)),
           'elapsed_seconds': time.perf_counter() - started, 'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'lgbm': lgb.__version__}}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
logger.info('OOF AUC=%.6f source-scaled AUC=%.6f elapsed=%.1fs', metrics['oof_auc'], metrics['source_scaled_oof_auc'], metrics['elapsed_seconds'])
display(metrics)

OOF AUC=0.830459 source-scaled AUC=0.845738 elapsed=40.3s


{'scope': 'monthly sampled smoke',
 'train_rows': 30000,
 'test_rows': 30002,
 'features': 772,
 'folds': [{'fold': 0,
   'months': [12],
   'auc': 0.8062923976229835,
   'best_iteration': 95},
  {'fold': 1,
   'months': [15],
   'auc': 0.8515145382176349,
   'best_iteration': 100},
  {'fold': 2, 'months': [13], 'auc': 0.843895256451587, 'best_iteration': 97},
  {'fold': 3, 'months': [17], 'auc': 0.8804722160418816, 'best_iteration': 41},
  {'fold': 4, 'months': [14], 'auc': 0.8759019607843137, 'best_iteration': 55},
  {'fold': 5,
   'months': [16],
   'auc': 0.8808173513370027,
   'best_iteration': 99}],
 'params': {'objective': 'binary',
  'boosting_type': 'gbdt',
  'metric': 'auc',
  'learning_rate': 0.007,
  'num_leaves': 256,
  'max_depth': -1,
  'tree_learner': 'serial',
  'feature_fraction': 0.5,
  'bagging_freq': 1,
  'bagging_fraction': 0.7,
  'max_bin': 255,
  'verbosity': -1,
  'seed': 42,
  'num_threads': 8},
 'rounds': 100,
 'target_mean': 'fold-local',
 'numeric_precision

### 저장된 결과에서 얻을 수 있는 판단
월별 표본 30,000 train/30,002 test, 6-fold, 100 rounds, 772피처입니다. raw OOF AUC=.830459, source-scaled=.845738입니다.
min-max 후 차이가 비교적 커도 재학습 개선은 아닙니다. 서로 다른 fold의 확률 범위가 pooled 순위에 영향을 준다는 신호로 읽으세요.
**스스로 설명해보세요:** 왜 단일 fold의 AUC는 유지되면서 전체 OOF AUC는 달라질 수 있나요? 왜 100-round 점수만으로 CatBoost와 최종 우열을 정하기 어렵나요?